# Pakistani Fashion Intelligence
## Step 4 — Peshawar Weather API

**Project:** Fashion Brand Competitive Price Analytics  
**Prepared by:** Aqib Hanif  
**Assigned By:** Mam Sumayyea Salahuddin  
**Institute:** Arfa Karim Incubation Center, Peshawar  

### Purpose of this notebook

In this step, I use the **Open-Meteo API** to add Peshawar weather information to my project.

The fashion prices in my project come from the official online stores. I use **Peshawar only as the regional and seasonal context** for my dashboard.

I will collect:

- Current temperature
- Weather condition
- Humidity
- Precipitation
- Wind speed
- Daily maximum temperature
- Daily minimum temperature
- Six months of seasonal weather context

Open-Meteo does not require an API key, so I can use it without creating an account.

## 1. Import the required libraries

I use:

- **requests** to call the API,
- **pandas** to store and summarize the weather data,
- **datetime** to calculate the seasonal date range.

In [ ]:
# I import the libraries that I need for my API step.

from datetime import date
from pathlib import Path

import pandas as pd
import requests

print("Libraries imported successfully.")

## 2. Set the Peshawar location

I use approximate coordinates for Peshawar, Pakistan.

These coordinates are only used to request weather information from Open-Meteo.

In [ ]:
# I keep the Peshawar location details in one place.

CITY = "Peshawar"
REGION = "Peshawar, Pakistan"

LATITUDE = 34.0151
LONGITUDE = 71.5249
TIMEZONE = "Asia/Karachi"

print("Region:", REGION)
print("Latitude:", LATITUDE)
print("Longitude:", LONGITUDE)

## 3. Create a simple weather-code function

Open-Meteo returns a numeric weather code.

I convert the main codes into easy weather names so the dashboard can show text such as **Clear**, **Partly Cloudy**, or **Rain**.

In [ ]:
# I convert Open-Meteo weather codes into simple descriptions.

def weather_description(code):

    if code == 0:
        return "Clear"

    if code in [1, 2]:
        return "Partly Cloudy"

    if code == 3:
        return "Overcast"

    if code in [45, 48]:
        return "Fog"

    if code in [51, 53, 55, 56, 57]:
        return "Drizzle"

    if code in [61, 63, 65, 66, 67]:
        return "Rain"

    if code in [71, 73, 75, 77]:
        return "Snow"

    if code in [80, 81, 82]:
        return "Rain Showers"

    if code in [85, 86]:
        return "Snow Showers"

    if code in [95, 96, 99]:
        return "Thunderstorm"

    return "Unknown" 

## 4. Request the current Peshawar weather

I call the Open-Meteo forecast endpoint and request only the fields that I need for the dashboard.

In [ ]:
# I prepare the Open-Meteo URL for current Peshawar weather.

current_url = "https://api.open-meteo.com/v1/forecast"

current_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "current": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "precipitation,"
        "weather_code,"
        "wind_speed_10m"
    ),
    "daily": (
        "temperature_2m_max,"
        "temperature_2m_min"
    ),
    "timezone": TIMEZONE,
    "forecast_days": 1,
}

print("Current weather API request is ready.")

In [ ]:
# I send my request to Open-Meteo and convert the response to JSON.

current_response = requests.get(
    current_url,
    params=current_params,
    timeout=30
)

current_response.raise_for_status()

current_data = current_response.json()

print("Current weather data received successfully.")

## 5. Prepare the current-weather table

I keep the current weather in one row because this will later become the weather card on my Excel dashboard.

In [ ]:
# I take the useful values from the API response.

current = current_data["current"]
daily = current_data["daily"]

weather_code = int(current["weather_code"])

current_weather_df = pd.DataFrame([
    {
        "Region": REGION,
        "Weather_Date": current["time"][:10],
        "Temperature_C": current["temperature_2m"],
        "Weather_Condition": weather_description(weather_code),
        "Weather_Code": weather_code,
        "Max_Temperature_C": daily["temperature_2m_max"][0],
        "Min_Temperature_C": daily["temperature_2m_min"][0],
        "Humidity_Percent": current["relative_humidity_2m"],
        "Precipitation_mm": current["precipitation"],
        "Wind_Speed_kmh": current["wind_speed_10m"],
        "Source": "Open-Meteo API",
    }
])

current_weather_df

## 6. Select the seasonal period

For the seasonal context chart, I use the **previous six complete months**.

I do this instead of using future weather because I want the seasonal chart to be based on completed historical observations.

In [ ]:
# I calculate the previous six complete months automatically.

today = pd.Timestamp.today().normalize()

# The last completed month ends one day before the current month starts.
last_complete_month_end = today.replace(day=1) - pd.Timedelta(days=1)

# I go back five more months to create a six-month period.
season_start = (
    last_complete_month_end
    .to_period("M")
    .start_time
    - pd.DateOffset(months=5)
)

season_end = last_complete_month_end

START_DATE = season_start.strftime("%Y-%m-%d")
END_DATE = season_end.strftime("%Y-%m-%d")

print("Seasonal start date:", START_DATE)
print("Seasonal end date:", END_DATE)

## 7. Request historical Peshawar weather

I use the Open-Meteo historical archive endpoint.

For each day, I request:

- maximum temperature,
- minimum temperature,
- precipitation.

In [ ]:
# I prepare the historical weather API request.

archive_url = "https://archive-api.open-meteo.com/v1/archive"

archive_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "daily": (
        "temperature_2m_max,"
        "temperature_2m_min,"
        "precipitation_sum"
    ),
    "timezone": TIMEZONE,
}

print("Historical weather API request is ready.")

In [ ]:
# I send the historical API request.

archive_response = requests.get(
    archive_url,
    params=archive_params,
    timeout=30
)

archive_response.raise_for_status()

archive_data = archive_response.json()

print("Historical weather data received successfully.")

## 8. Convert daily weather into a DataFrame

I first create a daily table. After that, I summarize it month by month for the final dashboard chart.

In [ ]:
# I create a daily historical weather table.

daily_data = archive_data["daily"]

daily_weather_df = pd.DataFrame({
    "Date": pd.to_datetime(daily_data["time"]),
    "Max_Temperature_C": daily_data["temperature_2m_max"],
    "Min_Temperature_C": daily_data["temperature_2m_min"],
    "Precipitation_mm": daily_data["precipitation_sum"],
})

daily_weather_df.head()

## 9. Create the six-month seasonal summary

For each month, I calculate:

- average maximum temperature,
- average minimum temperature,
- total precipitation.

This table will support the **Seasonal Context (Peshawar)** chart in Excel.

In [ ]:
# I create a month field and summarize the daily observations.

daily_weather_df["Month"] = (
    daily_weather_df["Date"]
    .dt.to_period("M")
    .astype(str)
)

seasonal_weather_df = (
    daily_weather_df
    .groupby("Month", as_index=False)
    .agg(
        Avg_Max_Temperature_C=("Max_Temperature_C", "mean"),
        Avg_Min_Temperature_C=("Min_Temperature_C", "mean"),
        Total_Precipitation_mm=("Precipitation_mm", "sum"),
    )
)

# I round the numbers so they are easier to read in Excel.
seasonal_weather_df[
    [
        "Avg_Max_Temperature_C",
        "Avg_Min_Temperature_C",
        "Total_Precipitation_mm",
    ]
] = seasonal_weather_df[
    [
        "Avg_Max_Temperature_C",
        "Avg_Min_Temperature_C",
        "Total_Precipitation_mm",
    ]
].round(1)

seasonal_weather_df

## 10. Basic API data checks

Before saving the API output, I check that I received the expected number of records and that the main weather values are not missing.

In [ ]:
# I check the current weather table.

print("Current weather rows:", len(current_weather_df))
print("Current weather missing values:", current_weather_df.isna().sum().sum())

# I check the seasonal summary.
print("Seasonal months:", len(seasonal_weather_df))
print("Seasonal missing values:", seasonal_weather_df.isna().sum().sum())

## 11. Save the weather API files

I save two CSV files:

1. `04_current_peshawar_weather.csv` — for the weather card.
2. `04_peshawar_seasonal_context.csv` — for the seasonal chart.

These files will be imported into the final Excel workbook.

In [ ]:
# I save my API outputs for the Excel dashboard.

CURRENT_OUTPUT = Path("04_current_peshawar_weather.csv")
SEASONAL_OUTPUT = Path("04_peshawar_seasonal_context.csv")

current_weather_df.to_csv(
    CURRENT_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

seasonal_weather_df.to_csv(
    SEASONAL_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Current weather file:", CURRENT_OUTPUT)
print("Seasonal weather file:", SEASONAL_OUTPUT)

## 12. Step 4 conclusion

In this step, I successfully added an **API component** to my project using Open-Meteo.

I used Peshawar as the regional context and prepared:

- a current-weather summary,
- and a six-month seasonal weather summary.

I do **not** claim that Peshawar weather directly causes fashion prices to change. I use it only as useful seasonal context for the local presentation region.

### Next step

In Step 5, I will combine:

- processed fashion data,
- PostgreSQL results,
- and Peshawar API data

inside my final **interactive Excel dashboard**.